<a href="https://colab.research.google.com/github/SyedaMalaika75/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedaMalaika75/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My baseline prioritizes content that is still receiving search visibility but may need attention.

The first signal identifies stale but visible pages. Pages with at least 500 impressions and 180 or more days since their last update receive higher priority.

Reason code: STALE_VISIBLE_PAGE
Action label: REFRESH_CONTENT

The second signal identifies pages with strong impressions, an average position between 1 and 20, but a CTR below 0.5%.

Reason code: LOW_CTR_VISIBLE_PAGE
Action label: REVIEW_TITLE_AND_META

The rule uses only information available at the decision moment and excludes future-window, label-derived and identifying fields.

In [15]:
import pandas as pd
import numpy as np

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "SyedaMalaika75/flyrank-ml-internship/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())

display(df.head())
# Evaluation label is used only to check the signals.
# It is NOT used to calculate the baseline score.
df["decline_label"] = (df["trend_direction"] == "down").astype(int)

# --------------------------------------------------
# SIGNAL 1: STALE + VISIBLE
# --------------------------------------------------

stale_visible = (
    (df["days_since_last_update"] >= 180)
    & (df["impressions_90d"] >= 500)
)

signal_1_table = pd.DataFrame({
    "bucket": [
        "Does not meet rule",
        "Stale + visible"
    ],
    "n": [
        (~stale_visible).sum(),
        stale_visible.sum()
    ],
    "decline_rate_pct": [
        df.loc[~stale_visible, "decline_label"].mean() * 100,
        df.loc[stale_visible, "decline_label"].mean() * 100
    ]
}).round({"decline_rate_pct": 1})

print("SIGNAL 1 — STALE + VISIBLE")
display(signal_1_table)

signal_1_verdict = (
    "CONFIRMED"
    if signal_1_table.loc[1, "decline_rate_pct"]
       > signal_1_table.loc[0, "decline_rate_pct"]
    else "OPPOSITE"
)

print("Verdict:", signal_1_verdict)


# --------------------------------------------------
# SIGNAL 2: LOW CTR + VISIBILITY + GOOD POSITION
# --------------------------------------------------

# avg_position must be between 1 and 20.
# Position 0 represents unavailable ranking data.
eligible = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"].between(1, 20))
)

ctr_signal_frame = df.loc[eligible].copy()

# CTR is stored as percentage points.
# Therefore 0.5 means 0.5%, not 50%.
ctr_signal_frame["ctr_bucket"] = pd.cut(
    ctr_signal_frame["ctr"],
    bins=[-0.001, 0.5, 1.0, 2.0, np.inf],
    labels=["<0.5%", "0.5–<1%", "1–<2%", "2%+"],
    right=False
)

signal_2_table = (
    ctr_signal_frame
    .groupby("ctr_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        decline_rate_pct=(
            "decline_label",
            lambda values: round(values.mean() * 100, 1)
        ),
        median_impressions=("impressions_90d", "median")
    )
    .reset_index()
)

print("\nSIGNAL 2 — CTR AMONG VISIBLE PAGES IN POSITIONS 1–20")
display(signal_2_table)

low_ctr_rate = signal_2_table.loc[
    signal_2_table["ctr_bucket"] == "<0.5%",
    "decline_rate_pct"
].iloc[0]

other_ctr_rate = (
    ctr_signal_frame.loc[
        ctr_signal_frame["ctr"] >= 0.5,
        "decline_label"
    ].mean() * 100
)

signal_2_verdict = (
    "CONFIRMED"
    if low_ctr_rate > other_ctr_rate
    else "OPPOSITE"
)

print("Verdict:", signal_2_verdict)
print(f"Low-CTR decline rate: {low_ctr_rate:.1f}%")
print(f"Other eligible pages: {other_ctr_rate:.1f}%")


Dataset shape: (30000, 44)

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


SIGNAL 1 — STALE + VISIBLE


,bucket,n,decline_rate_pct
0,Does not meet rule,29983,54.2
1,Stale + visible,17,94.1


Verdict: CONFIRMED

SIGNAL 2 — CTR AMONG VISIBLE PAGES IN POSITIONS 1–20


,ctr_bucket,n,decline_rate_pct,median_impressions
0,<0.5%,9745,62.7,3021.0
1,0.5–<1%,1728,48.1,4935.0
2,1–<2%,489,44.8,4152.0
3,2%+,47,53.2,3579.0


Verdict: CONFIRMED
Low-CTR decline rate: 62.7%
Other eligible pages: 47.5%


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I use a transparent point-based baseline rather than fitted weights.

A stale and visible page receives 100 points. A visible page with a useful search position but CTR below 0.5% receives 60 points. Up to 10 additional points are used only as an impressions-based tie-breaker.

Each ranked item receives one reason code and one action label. The observed trend is used only to evaluate the ranking and is never used to calculate the score.

In [16]:
from pathlib import Path

# Only these observed decision-time fields calculate the score.
scoring_features = [
    "days_since_last_update",
    "impressions_90d",
    "ctr",
    "avg_position"
]

queue = df.copy()

queue["signal_stale_visible"] = (
    (queue["days_since_last_update"] >= 180)
    & (queue["impressions_90d"] >= 500)
)

queue["signal_low_ctr_visible"] = (
    (queue["impressions_90d"] >= 500)
    & (queue["avg_position"].between(1, 20))
    & (queue["ctr"] < 0.5)
)

# Transparent hand-written score.
queue["volume_tiebreak"] = (
    np.minimum(queue["impressions_90d"] / 5000, 1.0) * 10
)

queue["baseline_score"] = (
    queue["signal_stale_visible"].astype(int) * 100
    + queue["signal_low_ctr_visible"].astype(int) * 60
    + queue["volume_tiebreak"]
).round(2)

both_signals = (
    queue["signal_stale_visible"]
    & queue["signal_low_ctr_visible"]
)

stale_only = (
    queue["signal_stale_visible"]
    & ~queue["signal_low_ctr_visible"]
)

ctr_only = (
    ~queue["signal_stale_visible"]
    & queue["signal_low_ctr_visible"]
)

queue["reason_code"] = np.select(
    [
        both_signals,
        stale_only,
        ctr_only
    ],
    [
        "STALE_AND_LOW_CTR",
        "STALE_VISIBLE_PAGE",
        "LOW_CTR_VISIBLE_PAGE"
    ],
    default="LOW_PRIORITY"
)

queue["action_label"] = np.select(
    [
        both_signals,
        stale_only,
        ctr_only
    ],
    [
        "REFRESH_AND_REVIEW_SNIPPET",
        "REFRESH_CONTENT",
        "REVIEW_TITLE_AND_META"
    ],
    default="MONITOR"
)

# Rank from highest to lowest priority.
queue = queue.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

queue.insert(
    0,
    "rank",
    np.arange(1, len(queue) + 1)
)

# Honest evaluation.
# decline_label is the outcome only, not a scoring input.
def precision_at_k(frame, k):
    return frame.head(k)["decline_label"].mean()

base_rate = queue["decline_label"].mean()
precision_20 = precision_at_k(queue, 20)
precision_50 = precision_at_k(queue, 50)

print("Base decline rate:", round(base_rate, 3))
print("Precision@20:", round(precision_20, 3))
print("Precision@50:", round(precision_50, 3))

output_columns = [
    "rank",
    "content_id",
    "baseline_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

# Write the required CSV.
output_directory = Path("work/outputs")
output_directory.mkdir(parents=True, exist_ok=True)

output_path = (
    output_directory
    / "baseline_action_score.csv"
)

queue[output_columns].to_csv(
    output_path,
    index=False
)

print("CSV written to:", output_path)

display(
    queue[output_columns].head(20)
)


Base decline rate: 0.542
Precision@20: 0.9
Precision@50: 0.6
CSV written to: work/outputs/baseline_action_score.csv


,rank,content_id,baseline_score,reason_code,action_label,impressions_90d,days_since_last_update,ctr,avg_position
0,1,content_cf56e2e2e282,170.00,STALE_AND_LOW_CTR,REFRESH_AND_REVIEW_SNIPPET,61678,194,0.15,19.7
1,2,content_0a91db491d14,170.00,STALE_AND_LOW_CTR,REFRESH_AND_REVIEW_SNIPPET,13299,193,0.49,10.5
2,3,content_c2d929d83eaa,170.00,STALE_AND_LOW_CTR,REFRESH_AND_REVIEW_SNIPPET,7558,193,0.20,17.9
3,4,content_fe16a55cd13d,169.11,STALE_AND_LOW_CTR,REFRESH_AND_REVIEW_SNIPPET,4556,194,0.33,16.4
4,5,content_928af3e22c80,163.39,STALE_AND_LOW_CTR,REFRESH_AND_REVIEW_SNIPPET,1697,193,0.12,15.8
5,6,content_e3ff1b093148,162.82,STALE_AND_LOW_CTR,REFRESH_AND_REVIEW_SNIPPET,1408,183,0.28,7.8
6,7,content_7f116ae1f6f5,161.91,STALE_AND_LOW_CTR,REFRESH_AND_REVIEW_SNIPPET,954,301,0.42,9.0
7,8,content_77d4d5930e5e,161.66,STALE_AND_LOW_CTR,REFRESH_AND_REVIEW_SNIPPET,828,194,0.24,18.6
8,9,content_72496874f806,161.64,STALE_AND_LOW_CTR,REFRESH_AND_REVIEW_SNIPPET,821,301,0.24,5.8
9,10,content_6226ee6adc91,161.09,STALE_AND_LOW_CTR,REFRESH_AND_REVIEW_SNIPPET,545,183,0.18,17.8


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the first 20 ranked items rather than treating the score as automatically correct.

Confidence is highest when both transparent signals agree. Each recommendation also includes a realistic condition that could make the recommendation wrong, such as evergreen content, unusual query intent, SERP features or naturally low-click searches.

In [17]:
review = queue.head(20).copy()

review["confidence_note"] = np.select(
    [
        (
            review["signal_stale_visible"]
            & review["signal_low_ctr_visible"]
        ),
        review["signal_stale_visible"],
        review["signal_low_ctr_visible"]
    ],
    [
        "High: both transparent signals agree.",
        "Medium-high: stale and still visible.",
        "Medium: strong visibility but weak CTR."
    ],
    default="Low: volume tie-break only."
)

review["what_would_make_it_wrong"] = np.select(
    [
        (
            review["signal_stale_visible"]
            & review["signal_low_ctr_visible"]
        ),
        review["signal_stale_visible"],
        review["signal_low_ctr_visible"]
    ],
    [
        (
            "The topic may be evergreen, or SERP/query "
            "intent may naturally suppress CTR."
        ),
        (
            "The content may be evergreen and still fully "
            "accurate despite its age."
        ),
        (
            "Low CTR may be normal because of SERP features, "
            "query mix or non-click intent."
        )
    ],
    default=(
        "The item may have no actionable issue "
        "beyond high volume."
    )
)

review_columns = [
    "rank",
    "content_id",
    "action_label",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

print("TOP-20 HUMAN REVIEW")

display(
    review[review_columns]
)


TOP-20 HUMAN REVIEW


,rank,content_id,action_label,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,REFRESH_AND_REVIEW_SNIPPET,STALE_AND_LOW_CTR,High: both transparent signals agree.,"The topic may be evergreen, or SERP/query inte..."
1,2,content_0a91db491d14,REFRESH_AND_REVIEW_SNIPPET,STALE_AND_LOW_CTR,High: both transparent signals agree.,"The topic may be evergreen, or SERP/query inte..."
2,3,content_c2d929d83eaa,REFRESH_AND_REVIEW_SNIPPET,STALE_AND_LOW_CTR,High: both transparent signals agree.,"The topic may be evergreen, or SERP/query inte..."
3,4,content_fe16a55cd13d,REFRESH_AND_REVIEW_SNIPPET,STALE_AND_LOW_CTR,High: both transparent signals agree.,"The topic may be evergreen, or SERP/query inte..."
4,5,content_928af3e22c80,REFRESH_AND_REVIEW_SNIPPET,STALE_AND_LOW_CTR,High: both transparent signals agree.,"The topic may be evergreen, or SERP/query inte..."
5,6,content_e3ff1b093148,REFRESH_AND_REVIEW_SNIPPET,STALE_AND_LOW_CTR,High: both transparent signals agree.,"The topic may be evergreen, or SERP/query inte..."
6,7,content_7f116ae1f6f5,REFRESH_AND_REVIEW_SNIPPET,STALE_AND_LOW_CTR,High: both transparent signals agree.,"The topic may be evergreen, or SERP/query inte..."
7,8,content_77d4d5930e5e,REFRESH_AND_REVIEW_SNIPPET,STALE_AND_LOW_CTR,High: both transparent signals agree.,"The topic may be evergreen, or SERP/query inte..."
8,9,content_72496874f806,REFRESH_AND_REVIEW_SNIPPET,STALE_AND_LOW_CTR,High: both transparent signals agree.,"The topic may be evergreen, or SERP/query inte..."
9,10,content_6226ee6adc91,REFRESH_AND_REVIEW_SNIPPET,STALE_AND_LOW_CTR,High: both transparent signals agree.,"The topic may be evergreen, or SERP/query inte..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The baseline is decision support, not proof that every selected page truly needs an intervention.

I treat a top-ranked item as a weak pick when its observed direction was not declining. Such cases may represent evergreen content, normal CTR behaviour, SERP effects or pages that remain accurate despite their age.

The score uses only freshness, impressions, CTR and observed search position. IDs, product flags, trend fields and outcome-derived information are excluded.

In [18]:
# Find top-ranked rows where the observed outcome was not declining.
weak_picks = review.loc[
    review["decline_label"] == 0
].copy()

weak_picks["why_it_looks_weak"] = np.select(
    [
        (
            weak_picks["signal_stale_visible"]
            & weak_picks["signal_low_ctr_visible"]
        ),
        weak_picks["signal_stale_visible"],
        weak_picks["signal_low_ctr_visible"]
    ],
    [
        (
            "Both signals fired, but the observed trend was "
            "not down. Evergreen content or SERP effects may "
            "explain the recommendation."
        ),
        (
            "Age and visibility fired, but the observed trend "
            "was not down. Staleness alone may over-prioritize "
            "evergreen pages."
        ),
        (
            "CTR was low despite visibility, but the observed "
            "trend was not down. The query mix may make low "
            "CTR normal."
        )
    ],
    default=(
        "The score was driven mainly by volume rather "
        "than a clear action signal."
    )
)

print(
    "WEAK PICKS FOUND IN THE TOP 20:",
    len(weak_picks)
)

display(
    weak_picks[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action_label",
            "trend_direction",
            "why_it_looks_weak"
        ]
    ]
)

# --------------------------------------------------
# LEAKAGE CHECK
# --------------------------------------------------

banned_exact = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "health_score",
    "ai_opportunity"
}

banned_prefixes = (
    "needs_",
    "is_"
)

leaked_inputs = [
    column
    for column in scoring_features
    if (
        column in banned_exact
        or column.startswith(banned_prefixes)
    )
]

print("Scoring features:", scoring_features)
print("Leaked inputs detected:", leaked_inputs)

assert not leaked_inputs, (
    "Leakage detected in scoring inputs."
)

assert "trend_direction" not in scoring_features
assert "trend_pct" not in scoring_features
assert "content_id" not in scoring_features
assert "client_id" not in scoring_features

print(
    "Leakage check PASSED: outcome fields and IDs "
    "were not used to calculate the score."
)


WEAK PICKS FOUND IN THE TOP 20: 2


,rank,content_id,baseline_score,reason_code,action_label,trend_direction,why_it_looks_weak
15,16,content_bdbec75c1148,102.63,STALE_VISIBLE_PAGE,REFRESH_CONTENT,stable,"Age and visibility fired, but the observed tre..."
18,19,content_aaef01a50def,70.00,LOW_CTR_VISIBLE_PAGE,REVIEW_TITLE_AND_META,stable,"CTR was low despite visibility, but the observ..."


Scoring features: ['days_since_last_update', 'impressions_90d', 'ctr', 'avg_position']
Leaked inputs detected: []
Leakage check PASSED: outcome fields and IDs were not used to calculate the score.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.